# Prepare a company-event timeline

## Goal

Output deduplicated event versions, defensible availability times, the next full session and review flags, without abnormal-return calculations.

This notebook uses synthetic teaching data, not a paper replication or production observations.

## Setup

Use a Python 3.10+ kernel and run all cells in order. Computation uses only the standard library, without keys, networking or extra data files. Open in an existing Jupyter environment.

Embedded inputs match inputs.json in the same download directory. Edit args in the next cell to experiment; preserve explicit times and units.

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"company-event-timeline\",\"identity\":\"synthetic\",\"args\":[[{\"id\":\"DEMO-A\",\"version\":\"v1\",\"publishedAt\":\"2025-01-03T18:00:00+08:00\",\"firstSeenAt\":\"2025-01-03T18:02:00+08:00\"},{\"id\":\"DEMO-A\",\"version\":\"v1\",\"publishedAt\":\"2025-01-03T18:00:00+08:00\",\"firstSeenAt\":\"2025-01-03T18:02:00+08:00\"},{\"id\":\"DEMO-B\",\"version\":\"v1\",\"publishedAt\":\"2025-01-06\",\"firstSeenAt\":\"2025-01-06T10:00:00+08:00\"}],[\"2025-01-03T09:30:00+08:00\",\"2025-01-06T09:30:00+08:00\",\"2025-01-07T09:30:00+08:00\"]],\"expected\":[{\"id\":\"DEMO-A\",\"version\":\"v1\",\"publishedAt\":\"2025-01-03T18:00:00+08:00\",\"firstSeenAt\":\"2025-01-03T18:02:00+08:00\",\"availableAt\":\"2025-01-03T10:02:00.000Z\",\"sessionOpen\":\"2025-01-06T09:30:00+08:00\",\"status\":\"aligned\"},{\"id\":\"DEMO-B\",\"version\":\"v1\",\"publishedAt\":\"2025-01-06\",\"firstSeenAt\":\"2025-01-06T10:00:00+08:00\",\"sessionOpen\":null,\"status\":\"needs_review\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## Steps

### 1. Freeze the event definition

An announcement, a news report and a corporate action's effective date are different objects. Define which one you need, preserving source links, content hashes, company mapping and revisions. Syndicated copies must not automatically become independent events.

### 2. Establish availability

The example takes the later of publication and first observation. Date-only records become needs_review rather than guessed pre- or post-market releases. Require timezone-aware timestamps; do not sort UTC and Shanghai local times as raw strings.

### 3. Use a conservative daily mapping

Choose the first session opening strictly after availability. A Friday after-close release maps to the next listed session; an intraday release also maps to the next full session. This is a declared daily convention, not an exchange rule or a method for immediate intraday reactions.

### 4. Keep unresolved records visible

Keep one exact duplicate; conflicting versions stop processing. Events beyond calendar coverage become outside_calendar and wait for a longer calendar, rather than guessing the next weekday. Keep unresolved records distinct so missing timing is not hidden.

### Method and assumptions

- Weekdays do not replace exchange holidays, closures or security-specific suspensions.
- First capture is evidence for your data chain, not the entire market's first knowledge.
- Event alignment does not establish causality; confounding events and selection remain research questions.

In [ ]:
def align_events(events, session_opens):
    """Map an event to the first opening strictly after its availability."""
    from datetime import datetime, timezone
    import re

    def parse(value):
        if not isinstance(value, str) or not re.search(r"T.*(Z|[+-]\d{2}:\d{2})$", value):
            return None
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            return None

    if not session_opens or any(parse(value) is None for value in session_opens):
        raise ValueError("invalid_calendar")
    sessions = sorted(set(session_opens), key=parse)
    seen, output = {}, []
    for event in events:
        if not event.get("id") or not event.get("version"):
            raise ValueError("missing_event_identity")
        key = (event["id"], event["version"])
        if key in seen:
            if seen[key] != event:
                raise ValueError("conflicting_event_version")
            continue
        seen[key] = dict(event)
        published, observed = parse(event.get("publishedAt")), parse(event.get("firstSeenAt"))
        if published is None or observed is None:
            output.append(dict(event, sessionOpen=None, status="needs_review"))
            continue
        available = max(published, observed)
        session = next((value for value in sessions if parse(value) > available), None)
        timestamp = available.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
        output.append(dict(event, availableAt=timestamp, sessionOpen=session,
                           status="aligned" if session else "outside_calendar"))
    return output


### Run the sample

Three synthetic inputs become two records: DEMO-A maps to the 2025-01-06 open; date-only DEMO-B remains needs_review with no session.

In [ ]:
result = align_events(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Checks

Compare every row with the browser example's expected output. After editing inputs, a failed assertion may be expected: explain the difference before changing the check.

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("Passed: output matches the synthetic browser example.")

## Next steps

Before real data, confirm grants, fields, schema_major, windows and provenance using authenticated GET /v1/catalog, then map the actual contract. Candidate IDs below do not guarantee availability or historical completeness. API as_of is not a historical filing-version guarantee. Validate again after substituting real inputs; the synthetic pass does not transfer.

- `cn.dataset.anns_d`
- `cn.market.trade_calendar`

### References

- [MacKinlay: event-study methods](https://www.jstor.org/stable/2729691)
- [Tushare: security identity and listing status](https://tushare.pro/document/2?doc_id=25)

[Back to tutorial](https://tradingdatas.com/recipes/company-event-timeline/)